In [0]:
from pyspark.sql import functions as F

# Load tables
workflow = spark.table("fct_workflow_events")

applications = (
    spark.table("bronze_applications")
    .select(
        F.trim(F.col("application_id")).alias("application_id"),
        F.trim(F.col("job_id")).alias("job_id"),
        F.trim(F.col("candidate_id")).alias("candidate_id"),
        F.trim(F.col("apply_date")).alias("apply_date")
    )
    .dropDuplicates(["application_id"])
)

# -----------------------------
# Step 1: Hired date
# -----------------------------
hired_data = (
    workflow
    .filter(F.col("new_status") == "Hired")
    .groupBy("application_id")
    .agg(F.min("event_timestamp").alias("hired_date"))
)

# -----------------------------
# Step 2: Current status
# -----------------------------
# Get latest timestamp per application
latest_event = (
    workflow
    .groupBy("application_id")
    .agg(F.max("event_timestamp").alias("latest_timestamp"))
)

# Join back to get corresponding status
current_status_data = (
    latest_event
    .join(
        workflow,
        (latest_event.application_id == workflow.application_id) &
        (latest_event.latest_timestamp == workflow.event_timestamp),
        "left"
    )
    .select(
        latest_event.application_id,
        F.col("new_status").alias("current_status")
    )
)

# -----------------------------
# Step 3: Join everything
# -----------------------------
fct_applications = (
    applications
    .join(hired_data, on="application_id", how="left")
    .join(current_status_data, on="application_id", how="left")
)

# -----------------------------
# Step 4: is_hired flag
# -----------------------------
fct_applications = fct_applications.withColumn(
    "is_hired",
    F.when(F.col("hired_date").isNotNull(), 1).otherwise(0)
)

# -----------------------------
# Step 5: Write table
# -----------------------------
fct_applications.write.format("delta").mode("overwrite").saveAsTable("fct_applications")

In [0]:
spark.sql('SELECT COUNT(*) FROM fct_applications;').show()

spark.sql('SELECT application_id, COUNT(*) FROM fct_applications GROUP BY application_id HAVING COUNT(*) > 1;').show()

spark.sql('SELECT * FROM fct_applications WHERE is_hired = 1 LIMIT 10;').show()